# 📊 Meridian MMM Tool
### Marketing Mix Modeling — Powered by Google Meridian

**Οδηγίες:**
1. Πάτα `Runtime` → `Change runtime type` → επίλεξε **T4 GPU**
2. Τρέξε τα cells ένα-ένα από πάνω προς τα κάτω
3. Σε κάθε βήμα θα σε καθοδηγεί το εργαλείο

---

In [ ]:
# ============================================================
# CELL 1 — Εγκατάσταση
# ============================================================
!pip install google-meridian openpyxl --quiet
print('✅ Έτοιμο!')

In [ ]:
# ============================================================
# CELL 2 — Ανέβασε τα αρχεία σου
# ============================================================
from google.colab import files
import pandas as pd
import numpy as np
import os

print('📁 Ανέβασε τα αρχεία σου (CSV ή Excel).')
print('   Μπορείς να ανεβάσεις 1 ή περισσότερα αρχεία.')
print()

uploaded = files.upload()

# Φόρτωσε όλα τα αρχεία
all_dataframes = {}

for filename, content in uploaded.items():
    print(f'\n📂 Διαβάζω: {filename}')
    try:
        if filename.endswith('.csv'):
            df = pd.read_csv(filename)
            all_dataframes[filename] = {'df': df, 'sheet': None}
            print(f'   ✅ {len(df)} γραμμές, {len(df.columns)} columns')
        elif filename.endswith(('.xlsx', '.xls')):
            xl = pd.ExcelFile(filename)
            print(f'   📋 Sheets: {xl.sheet_names}')
            for sheet in xl.sheet_names:
                df = pd.read_excel(filename, sheet_name=sheet)
                key = f'{filename} [{sheet}]'
                all_dataframes[key] = {'df': df, 'sheet': sheet}
                print(f'   ✅ Sheet "{sheet}": {len(df)} γραμμές, {len(df.columns)} columns')
    except Exception as e:
        print(f'   ❌ Σφάλμα: {e}')

print(f'\n✅ Φορτώθηκαν {len(all_dataframes)} πίνακες δεδομένων.')

In [ ]:
# ============================================================
# CELL 3 — Εμφάνιση όλων των columns
# ============================================================
print('=' * 60)
print('📋 ΟΛΑ ΤΑ COLUMNS ΑΠΟ ΤΑ ΑΡΧΕΙΑ ΣΟΥ')
print('=' * 60)

all_columns = []

for source, info in all_dataframes.items():
    df = info['df']
    print(f'\n📂 {source}:')
    for col in df.columns:
        sample = df[col].dropna().head(3).tolist()
        dtype = str(df[col].dtype)
        print(f'   • {col}')
        print(f'     Τύπος: {dtype} | Δείγμα: {sample}')
        all_columns.append({'source': source, 'column': col, 'dtype': dtype, 'sample': sample})

print(f'\n📊 Σύνολο columns: {len(all_columns)}')
print('\n➡️  Τρέξε το επόμενο cell για να ορίσεις τι είναι το καθένα.')

In [ ]:
# ============================================================
# CELL 4 — Επιβεβαίωση columns από τον χρήστη
# ============================================================
print('=' * 60)
print('⚙️  ΟΡΙΣΜΟΣ COLUMNS')
print('=' * 60)
print('''
Για κάθε column θα σε ρωτήσω τι είναι.
Επίλεξε έναν αριθμό:

  1 = 📅 DATE (ημερομηνία / περίοδος)
  2 = 💰 KPI / REVENUE (το κύριο metric που θέλεις να μοντελοποιήσεις)
  3 = 📢 MEDIA SPEND (κόστος διαφήμισης σε €)
  4 = 👁  MEDIA IMPRESSIONS (εμφανίσεις / εντυπώσεις)
  5 = 🎛  CONTROL VARIABLE (εξωγενής παράγοντας πχ εποχικότητα)
  6 = ⏭  ΠΑΡΑΛΕΙΨΕ (δεν χρειάζεται για το μοντέλο)
''')

column_mapping = []
type_map = {
    '1': 'date',
    '2': 'kpi',
    '3': 'media_spend',
    '4': 'media_impressions',
    '5': 'control',
    '6': 'skip'
}

type_labels = {
    '1': '📅 DATE',
    '2': '💰 KPI/REVENUE',
    '3': '📢 MEDIA SPEND',
    '4': '👁  IMPRESSIONS',
    '5': '🎛  CONTROL',
    '6': '⏭  ΠΑΡΑΛΕΙΨΕ'
}

for info in all_columns:
    print(f'\n┌─────────────────────────────────────────')
    print(f'│ Column: {info["column"]}')
    print(f'│ Από: {info["source"]}')
    print(f'│ Δείγμα τιμών: {info["sample"]}')
    print(f'└─────────────────────────────────────────')

    while True:
        choice = input('   Τι είναι αυτό; [1-6]: ').strip()
        if choice in type_map:
            col_type = type_map[choice]
            label = type_labels[choice]

            channel_name = None
            if col_type in ['media_spend', 'media_impressions']:
                channel_name = input(f'   Όνομα channel (πχ Google Search, Meta, TV): ').strip()

            column_mapping.append({
                'source': info['source'],
                'column': info['column'],
                'type': col_type,
                'channel_name': channel_name
            })
            print(f'   ✅ Ορίστηκε ως: {label}' + (f' → {channel_name}' if channel_name else ''))
            break
        else:
            print('   ⚠️  Παρακαλώ επίλεξε αριθμό από 1 έως 6.')

print('\n' + '=' * 60)
print('✅ ΟΛΟΚΛΗΡΩΣΗ ΟΡΙΣΜΟΥ COLUMNS')
print('=' * 60)

# Εμφάνιση περίληψης
print('\n📋 Περίληψη:')
for m in column_mapping:
    if m['type'] != 'skip':
        ch = f" → {m['channel_name']}" if m['channel_name'] else ''
        print(f"   {m['type'].upper()}: {m['column']}{ch}")

print('\n➡️  Τρέξε το επόμενο cell για να ορίσεις το date range.')

In [ ]:
# ============================================================
# CELL 5 — Date range
# ============================================================
print('=' * 60)
print('📅 ΟΡΙΣΜΟΣ DATE RANGE')
print('=' * 60)

date_start = input('\nΑρχή περιόδου (πχ 2022-01-01): ').strip()
date_end = input('Τέλος περιόδου (πχ 2024-12-31): ').strip()

print(f'\n✅ Περίοδος ανάλυσης: {date_start} → {date_end}')
print('\n➡️  Τρέξε το επόμενο cell για να ετοιμαστούν τα δεδομένα.')

In [ ]:
# ============================================================
# CELL 6 — Προετοιμασία δεδομένων για Meridian
# ============================================================
print('⚙️  Προετοιμασία δεδομένων...')

# Βρες date, kpi, spend, impressions columns
date_col_info = next((m for m in column_mapping if m['type'] == 'date'), None)
kpi_col_info = next((m for m in column_mapping if m['type'] == 'kpi'), None)
spend_cols = [m for m in column_mapping if m['type'] == 'media_spend']
impr_cols = [m for m in column_mapping if m['type'] == 'media_impressions']
control_cols = [m for m in column_mapping if m['type'] == 'control']

# Φτιάξε ένα ενιαίο dataframe
def get_col(mapping_info):
    src = mapping_info['source']
    col = mapping_info['column']
    return all_dataframes[src]['df'][col]

# Ξεκίνα με date και kpi
date_series = get_col(date_col_info)
kpi_series = get_col(kpi_col_info)

merged = pd.DataFrame({
    'date': pd.to_datetime(date_series),
    'kpi': pd.to_numeric(kpi_series, errors='coerce')
})

# Πρόσθεσε spend columns
media_channels = {}
media_spend_map = {}
for m in spend_cols:
    col_name = f"{m['channel_name']}_spend"
    merged[col_name] = pd.to_numeric(get_col(m), errors='coerce').fillna(0)
    media_spend_map[col_name] = m['channel_name']

# Πρόσθεσε impressions columns
for m in impr_cols:
    col_name = f"{m['channel_name']}_impressions"
    merged[col_name] = pd.to_numeric(get_col(m), errors='coerce').fillna(0)
    media_channels[col_name] = m['channel_name']

# Πρόσθεσε controls
control_names = []
for m in control_cols:
    col_name = m['column']
    merged[col_name] = pd.to_numeric(get_col(m), errors='coerce').fillna(0)
    control_names.append(col_name)

# Φιλτράρισμα date range
merged = merged[(merged['date'] >= date_start) & (merged['date'] <= date_end)]
merged = merged.dropna(subset=['kpi'])
merged = merged.sort_values('date').reset_index(drop=True)

# Αποθήκευση
merged.to_csv('/content/prepared_data.csv', index=False)

print(f'\n✅ Δεδομένα έτοιμα!')
print(f'   📅 Εγγραφές: {len(merged)}')
print(f'   📅 Από: {merged["date"].min()} → {merged["date"].max()}')
print(f'   💰 KPI: kpi ({merged["kpi"].mean():.0f} μέσος όρος)')
print(f'   📢 Spend channels: {list(media_spend_map.keys())}')
print(f'   👁  Impression channels: {list(media_channels.keys())}')
print()
print('➡️  Τρέξε το επόμενο cell για να ξεκινήσει το Meridian model.')

In [ ]:
# ============================================================
# CELL 7 — Εκτέλεση Meridian
# ============================================================
print('🚀 Εκκίνηση Meridian MMM...')
print('⏱  Αναμενόμενος χρόνος: 15-30 λεπτά με GPU, 40-60 χωρίς')
print()

from meridian.data import load
from meridian.model import model as mmm_model
from meridian.analysis import analyzer as mmm_analyzer
from meridian.analysis import optimizer as mmm_optimizer

# Χτίσε το column mapping για Meridian
coord_to_columns = load.CoordToColumns(
    time='date',
    kpi='kpi',
    controls=control_names if control_names else None,
    media=list(media_channels.keys()) if media_channels else None,
    media_spend=list(media_spend_map.keys()),
)

loader = load.CsvDataLoader(
    csv_path='/content/prepared_data.csv',
    kpi_type='non_revenue',
    coord_to_columns=coord_to_columns,
    media_to_channel=media_channels if media_channels else None,
    media_spend_to_channel=media_spend_map,
)

print('📥 Φόρτωση δεδομένων...')
input_data = loader.load()
print('✅ Δεδομένα φορτώθηκαν!')

print('🧠 Εκπαίδευση μοντέλου...')
model = mmm_model.Meridian(input_data=input_data)
model.sample_prior(500)
model.sample_posterior(
    n_chains=4,
    n_adapt=500,
    n_burnin=500,
    n_keep=1000
)
print('✅ Μοντέλο εκπαιδεύτηκε!')

In [ ]:
# ============================================================
# CELL 8 — Ανάλυση & Report
# ============================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import base64
from io import BytesIO

print('📊 Δημιουργία αναφοράς...')

analysis = mmm_analyzer.Analyzer(model)

def fig_to_base64(fig):
    buf = BytesIO()
    fig.savefig(buf, format='png', dpi=150, bbox_inches='tight', facecolor='white')
    buf.seek(0)
    encoded = base64.b64encode(buf.read()).decode('utf-8')
    plt.close(fig)
    return encoded

plots = {}
tables = {}

try:
    plots['model_fit'] = fig_to_base64(analysis.plot_model_fit())
    print('  ✅ Model Fit')
except Exception as e:
    print(f'  ⚠️  Model Fit: {e}')

try:
    plots['roi'] = fig_to_base64(analysis.plot_roi_bar_chart())
    print('  ✅ ROI Chart')
except Exception as e:
    print(f'  ⚠️  ROI: {e}')

try:
    plots['decomposition'] = fig_to_base64(analysis.plot_contribution_waterfall_chart())
    print('  ✅ Decomposition')
except Exception as e:
    print(f'  ⚠️  Decomposition: {e}')

try:
    plots['response_curves'] = fig_to_base64(analysis.plot_response_curves())
    print('  ✅ Response Curves')
except Exception as e:
    print(f'  ⚠️  Response Curves: {e}')

try:
    plots['budget'] = fig_to_base64(analysis.plot_budget_allocation_vs_optimized())
    print('  ✅ Budget Optimization')
except Exception as e:
    print(f'  ⚠️  Budget: {e}')

try:
    tables['roi'] = analysis.roi_summary().to_html(classes='data-table', border=0)
except:
    tables['roi'] = ''

# Φτιάξε HTML report
html = f'''
<!DOCTYPE html><html><head><meta charset="UTF-8">
<title>MMM Report {date_start} to {date_end}</title>
<style>
body{{font-family:Arial,sans-serif;background:#f8f9fa;margin:0;padding:0;color:#202124}}
.header{{background:linear-gradient(135deg,#1a73e8,#0d47a1);color:white;padding:40px 48px}}
.header h1{{margin:0;font-size:32px}}
.header p{{margin:8px 0 0;opacity:.85}}
.container{{max-width:1100px;margin:0 auto;padding:36px 24px}}
.section{{background:white;border-radius:16px;padding:32px;margin-bottom:28px;box-shadow:0 2px 8px rgba(0,0,0,.08)}}
.section h2{{font-size:20px;color:#1a73e8;padding-bottom:14px;border-bottom:2px solid #e8f0fe;margin-top:0}}
.section img{{width:100%;border-radius:10px;margin-top:8px}}
.section p{{color:#5f6368;font-size:14px;line-height:1.6}}
table{{width:100%;border-collapse:collapse;margin-top:16px;font-size:14px}}
th{{background:#1a73e8;color:white;padding:12px 16px;text-align:left}}
td{{padding:11px 16px;border-bottom:1px solid #f1f3f4}}
.footer{{text-align:center;padding:32px;color:#9aa0a6;font-size:13px}}
</style></head><body>
<div class="header">
  <h1>📊 Marketing Mix Model Report</h1>
  <p>Περίοδος: {date_start} → {date_end} &nbsp;·&nbsp; Powered by Google Meridian</p>
</div>
<div class="container">
'''

sections = [
    ('model_fit', '📈 1. Model Fit — Predicted vs Actual', 'Πόσο καλά εξηγεί το μοντέλο τα ιστορικά δεδομένα.'),
    ('roi', '💰 2. ROI ανά Channel', 'Revenue ανά €1 που δαπανήθηκε σε κάθε channel.'),
    ('decomposition', '🔍 3. KPI Decomposition', 'Τι ποσοστό του KPI προήλθε από κάθε channel.'),
    ('response_curves', '📉 4. Response Curves', 'Diminishing returns ανά channel.'),
    ('budget', '🎯 5. Budget Optimization', 'Προτεινόμενη αναδιανομή budget για μέγιστο KPI.'),
]

for key, title, desc in sections:
    html += f'<div class="section"><h2>{title}</h2><p>{desc}</p>'
    if key in plots and plots[key]:
        html += f'<img src="data:image/png;base64,{plots[key]}">'
    if key == 'roi' and tables.get('roi'):
        html += tables['roi']
    html += '</div>'

html += '<div class="footer">Meridian MMM Tool · Google Meridian · ' + date_start + ' → ' + date_end + '</div>'
html += '</div></body></html>'

# Αποθήκευση report
with open('/content/mmm_report.html', 'w', encoding='utf-8') as f:
    f.write(html)

print('\n✅ Report δημιουργήθηκε!')
print('\n📥 Κατέβασε το report:')
from google.colab import files
files.download('/content/mmm_report.html')
print('✅ Έτοιμο! Άνοιξε το αρχείο mmm_report.html στον browser σου.')